# 00.05 — ε + explore vs exploit

<div><span style="display:inline-block;background:#FF6B9A;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EPSILON</span><span style="display:inline-block;background:#9C6ADE;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EXPLORE</span><span style="display:inline-block;background:#2F80ED;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EXPLOIT</span><span style="display:inline-block;background:#ECA82C;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">Q</span></div>

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
repo=Path.cwd()
if repo.name=="notebooks": repo=repo.parent
sys.path.insert(0,str(repo))
from lineworld import LineWorld
from learning_cards import *
legend()

<div style="border-left:10px solid #2F80ED;background:#16181c;color:#f4f4f5;padding:14px 18px;border-radius:9px;margin:10px 0 14px 0"><div style="font-size:12px;font-weight:900;letter-spacing:.12em;color:#2F80ED">EXPLOIT</div><div style="font-size:21px;font-weight:800;margin-top:4px">Exploit = pick the best-looking action</div><div style="font-size:15px;line-height:1.55;margin-top:8px">Greedy choice: <code>argmax(Q[s])</code>. Uses what the table already believes.</div></div>

In [ ]:
Q=np.zeros((6,2)); Q[3]=[.10,.40]
fig=q_heatmap(Q,"State 3: RIGHT currently looks better (.40 > .10)")
plt.show()
print("greedy action in state 3 =",int(np.argmax(Q[3])),"(RIGHT)")

<div style="border-left:10px solid #9C6ADE;background:#16181c;color:#f4f4f5;padding:14px 18px;border-radius:9px;margin:10px 0 14px 0"><div style="font-size:12px;font-weight:900;letter-spacing:.12em;color:#9C6ADE">EXPLORE</div><div style="font-size:21px;font-weight:800;margin-top:4px">Explore = pick a random action</div><div style="font-size:15px;line-height:1.55;margin-top:8px">If the agent only exploits, entries it never tries stay at their starting value forever. Exploring is how untried actions get evidence.</div></div>

<div style="border-left:10px solid #FF6B9A;background:#16181c;color:#f4f4f5;padding:14px 18px;border-radius:9px;margin:10px 0 14px 0"><div style="font-size:12px;font-weight:900;letter-spacing:.12em;color:#FF6B9A">EPSILON</div><div style="font-size:21px;font-weight:800;margin-top:4px">ε = probability of exploring on a step</div><div style="font-size:15px;line-height:1.55;margin-top:8px">Flip a biased coin every step: with probability ε take a random action, otherwise take argmax(Q[s]).</div></div>

In [ ]:
rng=np.random.default_rng(0); epsilon=.25; picks=[]
for _ in range(200):
    picks.append("explore" if rng.random()<epsilon else "exploit")
counts=pd.Series(picks).value_counts().reindex(["exploit","explore"])
plt.figure(figsize=(7,3)); plt.bar(counts.index,counts.values,color=[COLORS["EXPLOIT"],COLORS["EXPLORE"]]); plt.title(f"200 decisions with ε={epsilon}"); plt.ylabel("count"); plt.show()
print(counts.to_dict())

<div style="border-left:10px solid #ECA82C;background:#16181c;color:#f4f4f5;padding:14px 18px;border-radius:9px;margin:10px 0 14px 0"><div style="font-size:12px;font-weight:900;letter-spacing:.12em;color:#ECA82C">Q</div><div style="font-size:21px;font-weight:800;margin-top:4px">Why ε matters: greedy can get stuck</div><div style="font-size:15px;line-height:1.55;margin-top:8px">Start every Q at -0.5 (a pessimistic guess). In state 0, LEFT (action 0) wins the tie and its estimate only rises toward about -0.4, so it keeps beating the untried RIGHT at -0.5. A pure-greedy agent bumps into the wall until the time limit and never sees the goal. (With a table of zeros, the -0.02 step cost makes tried actions look worse than untried ones, so greedy explores by accident. Do not count on that.)</div></div>

In [ ]:
def train(epsilon,q_init=-.5,episodes=300,alpha=.2,gamma=.95,seed=7):
    rng=np.random.default_rng(seed); env=LineWorld(); Q=np.full((6,2),q_init); returns=[]
    for ep in range(episodes):
        obs,_=env.reset(); total=0
        while True:
            action=int(rng.integers(0,2)) if rng.random()<epsilon else int(np.argmax(Q[obs]))
            n,r,term,trunc,_=env.step(action); future=0 if term else np.max(Q[n])
            Q[obs,action]+=alpha*(r+gamma*future-Q[obs,action]); total+=r; obs=n
            if term or trunc: break
        returns.append(total)
    return Q,returns
plt.figure(figsize=(9,4))
for eps,key in [(0.0,"EXPLOIT"),(.25,"EPSILON"),(.5,"EXPLORE")]:
    _,ret=train(eps); plt.plot(pd.Series(ret).rolling(20,min_periods=1).mean(),label=f"ε={eps}",color=COLORS[key])
plt.xlabel("episode"); plt.ylabel("rolling return"); plt.title("Q starts at -0.5: ε=0 never finds the goal (return stays -0.6); exploration does"); plt.legend(); plt.show()

<div style="border-left:10px solid #FF6B9A;background:#16181c;color:#f4f4f5;padding:14px 18px;border-radius:9px;margin:10px 0 14px 0"><div style="font-size:12px;font-weight:900;letter-spacing:.12em;color:#FF6B9A">EPSILON</div><div style="font-size:21px;font-weight:800;margin-top:4px">Takeaway</div><div style="font-size:15px;line-height:1.55;margin-top:8px">Exploit uses the table. Explore fills in the table. ε sets the mix. Next: 00.06 walks one full episode line by line.</div></div>